# DRC posts

Control panel for the X recent-search pull. Recent search only returns the last 7 days. Each returned post costs about $0.005, including posts fetched past the cap on the last page.

Leave `RUN_FETCH` false to review the files already saved. Set it to true only when you want a new pull.

In [ ]:
from collections import Counter
from pathlib import Path

import fetch_recent as x

RUN_FETCH = False
TARGET = 1000
QUERY = x.QUERY

token = x.bearer_token()
print(f"token loaded ({len(token)} chars), query length {len(QUERY)}, target {TARGET}")

In [ ]:
if RUN_FETCH:
    collected = x.collect(token, QUERY, TARGET)
    summary = x.save_run(collected, QUERY, TARGET)
else:
    summary = x.latest_summary()
    print("using the latest saved run")

summary

In [ ]:
posts = x.read_jsonl(Path(summary["posts_file"]))
users = {user["id"]: user for user in x.read_jsonl(Path(summary["users_file"]))}
print(f"{len(posts)} posts, {len(users)} accounts")
print("languages:", summary["languages"])
print("per day:", summary["posts_per_day"])

In [ ]:
TERMS = {
    "M23": ("m23",),
    "Rwanda": ("rwanda",),
    "FARDC": ("fardc",),
    "displaced": ("déplac", "deplac", "displac"),
    "South Kivu": ("sud-kivu", "south kivu", "sud kivu"),
    "Goma": ("goma",),
    "Ituri": ("ituri",),
    "ADF": ("adf",),
    "Bukavu": ("bukavu",),
    "North Kivu": ("nord-kivu", "north kivu", "nord kivu"),
    "Uvira": ("uvira",),
    "MONUSCO": ("monusco",),
}

counts = Counter()
for post in posts:
    text = (post.get("text") or "").lower()
    for name, keys in TERMS.items():
        if any(key in text for key in keys):
            counts[name] += 1

counts.most_common()

In [ ]:
def engagement(post):
    metrics = post.get("public_metrics") or {}
    return metrics.get("like_count", 0) + metrics.get("retweet_count", 0) + metrics.get("reply_count", 0)

rows = []
for post in sorted(posts, key=engagement, reverse=True)[:15]:
    metrics = post.get("public_metrics") or {}
    account = users.get(post.get("author_id"), {})
    rows.append({
        "account": account.get("username"),
        "likes": metrics.get("like_count", 0),
        "reposts": metrics.get("retweet_count", 0),
        "when": (post.get("created_at") or "")[:16],
        "text": " ".join((post.get("text") or "").split())[:240],
    })

rows

## 30 / 60 / 90-day escalation forecast

For each hotspot: the probability of a **major escalation** within 30, 60 and 90 days of the pull. A major escalation is a single event with 10+ deaths, a locality changing hands, or 5,000+ newly displaced. "Any violence" is not the target because it is close to certain every month across the east.

How it works (`forecast.py`):
- Drop off-topic posts and history/anniversary posts.
- Tag each remaining post with hotspots, actors, and signals: event, control change, escalation, de-escalation.
- Count **unique authors** per signal, and count copy-pasted text once, so campaigns do not dominate.
- Baseline monthly hazard per hotspot × a signal multiplier that fades over about 3 weeks. Chatter therefore moves the 30-day number most, and the 90-day number drifts back toward the baseline.

The baselines are priors (`TIER_HAZARD`) until they are replaced with ACLED counts. The probabilities are uncalibrated until scored. Read **lift** (signal above baseline) as the early-warning part.

In [ ]:
import forecast as fc

result = fc.forecast(posts)
print(f"{result['usable_posts']} usable posts, dropped {result['dropped_off_topic']} off-topic and {result['dropped_historical']} historical")

[
    {
        "hotspot": row["hotspot"],
        "p30": row["p30"],
        "p60": row["p60"],
        "p90": row["p90"],
        "p30_lift": row["p30_lift"],
        "signals": f"ev {row['event']} / ctl {row['control_change']} / esc {row['escalation']} / de-esc {row['deescalation']}",
        "actors": ", ".join(row["actors"]),
    }
    for row in result["rows"]
]

In [ ]:
# Evidence behind the five hotspots with the biggest signal lift
for row in sorted(result["rows"], key=lambda row: row["p30_lift"], reverse=True)[:5]:
    print(f"\n{row['hotspot']}  p30 {row['p30']:.0%} (baseline {row['p30_baseline']:.0%}), multiplier x{row['signal_multiplier']}")
    for post in row["evidence"]:
        print(f"  {post['created_at'][:16]}  {post['text'][:220]}")

In [ ]:
# Save this run with resolve-by dates so it can be scored once outcomes are known
forecast_path = fc.save_forecast(result, summary)
print(forecast_path)

### Calibrating and scoring

- **Base rates:** export ACLED events for the DRC over the last 12 months, then `fc.forecast(posts, base_hazard=fc.base_rates_from_acled(path, months=12))`.
- **Scoring:** after a resolve-by date, record for each hotspot the days until its first major escalation (`None` if none), then call `fc.brier(forecast_path, outcomes)`. Lower is better. Compare against the baseline-only forecast to check whether the X signal adds anything.
- **Run weekly:** recent search only covers 7 days, so a weekly pull gives a time series of signals and forecasts.

In [ ]:
ACLED_CSV = None  # e.g. Path("data/acled_drc_last12m.csv")

if ACLED_CSV:
    base = fc.base_rates_from_acled(ACLED_CSV, months=12)
    calibrated = fc.forecast(posts, base_hazard=base)
    [(row["hotspot"], row["p30"], row["p60"], row["p90"]) for row in calibrated["rows"]]

## Pipeline runs

The scheduled path is `python pipeline.py run`. It writes a run folder with a run record, tagged posts, forecast, health checks and a one-page brief. This cell shows the latest one.

In [ ]:
from IPython.display import Markdown

latest_run = sorted((x.ROOT / "data" / "runs").glob("*/brief.md"))[-1]
print(latest_run.parent.name)
Markdown(latest_run.read_text())